In [387]:
import folium
import numpy as np
import pandas as pd
import webbrowser
from branca.element import Template, MacroElement

In [388]:
edges = (
    pd.read_csv("../data/edges.csv")
    .rename(columns=lambda x: x.strip())[["# source", "target", "distance"]]
    .drop_duplicates()
)
edges = edges.set_index(["# source", "target"])
nodes = pd.read_csv("../data/nodes.csv").rename(columns=lambda x: x.strip())[
    ["name", "city", "country", "latitude", "longitude"]
]
nodes = pd.merge(nodes, pd.read_csv("../data/continents.csv"), on="country")

# solwesi ma zle suradnice a zle distance tympadom: -12.170638697187435,26.36774594217987

In [389]:
class RouteMap:
    def __init__(self, center=[20, 0], zoom_start=2, tiles="cartodb positron"):
        """Inicializácia mapy s definovanými štýlmi."""
        self.m = folium.Map(
            location=center, zoom_start=zoom_start, tiles=tiles, prefer_canvas=True
        )
        self.route_color = "#546E7A"  # Jednotná šedomodrá pre hrany a ohraničenia

    def add_categorical_legend(self):
        """Pridá diskrétnu HTML legendu do pravého dolného rohu mapy."""
        template = """
        {% macro html(this, kwargs) %}
        <div id='maplegend' class='maplegend' 
            style='position: fixed; z-index:9999; border:2px solid #546E7A; background-color:rgba(255, 255, 255, 0.9);
            border-radius:10px; padding: 15px; font-size:14px; left: 20px; bottom: 30px; font-family: "Arial", sans-serif;
            box-shadow: 0 4px 10px rgba(0,0,0,0.3);'>
            
            <div style='font-weight: 900; margin-bottom: 10px; border-bottom: 1px solid #ccc; 
                        padding-bottom: 5px; text-transform: uppercase; font-size: 10pt;'>
                Legenda letísk
            </div>
            <div style='display: flex; flex-direction: column; gap: 8px;'>
                <div style='display: flex; align-items: center;'>
                    <i style='background:#DD0A0A; width:14px; height:14px; border-radius:50%; display:inline-block; margin-right:10px; border:1px solid white;'></i>
                    Východzí list (Out)
                </div>
                <div style='display: flex; align-items: center;'>
                    <i style='background:#096409; width:14px; height:14px; border-radius:50%; display:inline-block; margin-right:10px; border:1px solid white;'></i>
                    Vstupný list (In)
                </div>
                <div style='display: flex; align-items: center;'>
                    <i style='background:#D6B057; width:14px; height:14px; border-radius:50%; display:inline-block; margin-right:10px; border:1px solid white;'></i>
                    Obojsmerný list
                </div>
            </div>
        </div>
        {% endmacro %}
        """
        macro = MacroElement()
        macro._template = Template(template)
        self.m.get_root().add_child(macro)

    def add_nodes(self, df, show_text=False):
        """Pridá letiská a kontinenty ako body s popiskami."""
        continents_list = ["Africa", "Asia", "Europe", "North America", "Oceania", "South America"]

        for idx, r in df.iterrows():
            # Určenie farby výplne podľa stĺpca 'way'
            if "way" in df.columns:
                if r["way"] == "out":
                    node_fill = "#DD0A0A"
                elif r["way"] == "in":
                    node_fill = "#096409"
                else:
                    node_fill = "#D6B057"
            else:
                node_fill = "#DD0A0A"

            # 1. Bod letiska (CircleMarker)
            folium.CircleMarker(
                [r["latitude"], r["longitude"]],
                radius=8 if "way" in df.columns else 6,
                color="white",
                weight=2,
                fill=True,
                fill_color=node_fill,
                fill_opacity=0.8,
                zorder=100,
                tooltip=f"<div>{r['name']}</div>",
                popup=f"<div><strong>Name:</strong> {r['name']}<br><strong>City:</strong> {r['city']}</div>"
                if "in" not in df.columns else
                f"<div><strong>Continent:</strong> {r['name']}<br>\
                        <strong>In:</strong> {r['in']}<br>\
                        <strong>Out:</strong>{r['out']}<br>\
                        <strong>Leafs:</strong>{r['leaf']}<br>\
                  </div>",
            ).add_to(self.m)

            # 2. Textové vizitky (ak je povolené)
            if show_text:
                raw_name = r.get("city", r.get("name", str(idx)))
                name = raw_name.replace("International Airport", "").replace("Airport", "").strip().rstrip(",")
                
                is_continent = raw_name in continents_list
                bg_color = "#115E07" if is_continent else "white"
                text_color = "white" if is_continent else "black"
                badge_txt = r.get("country", "") if "in" not in df.columns else (
                    f"<div style='font-size: 10pt;'><strong>In: </strong> {r['in']}<br>\
                        <strong>Out: </strong>{r['out']}<br>\
                        <strong>Leafs: </strong>{r['leaf']}<br>\
                     </div>"
                )
                anchor = r.get("anchor", (79, 40))

                html_content = f"""
                    <div style="background: {bg_color}; color: {text_color}; border: 2px solid {self.route_color}; 
                                border-radius: 8px; width: 180px; box-shadow: 0 4px 8px rgba(0,0,0,0.2); 
                                overflow: hidden; display: flex; flex-direction: column; font-family: Arial;">
                        <div style="padding: 6px; text-align: center; font-size: 11pt; font-weight: 900; text-transform: uppercase;">
                            {name}
                        </div>
                        <div style="background: {self.route_color}; color: white; font-size: 8pt; font-weight: bold; 
                                    padding: 2px 0; text-align: center; text-transform: uppercase;">
                            {badge_txt}
                        </div>
                    </div>
                """

                folium.Marker(
                    [r["latitude"], r["longitude"]],
                    icon=folium.DivIcon(icon_size=(180, 80), icon_anchor=anchor, html=html_content),
                ).add_to(self.m)

    def add_route(self, df, value_show=True):
        """Pridá letecké trasy s ošetrením prechodu cez 180. poludník."""
        for _, row in df.iterrows():
            s_lat, s_lon = map(float, str(row["start"]).split(","))
            e_lat, e_lon = map(float, str(row["end"]).split(","))

            v_e_lon = e_lon
            needs_split = False
            
            if e_lon - s_lon > 180:
                v_e_lon -= 360
                needs_split = True
            elif e_lon - s_lon < -180:
                v_e_lon += 360
                needs_split = True

            # Vykreslenie čiary (so splitom pre antimeridián)
            if needs_split:
                edge_lon = -180.0 if v_e_lon < -180 else 180.0
                ratio = (edge_lon - s_lon) / (v_e_lon - s_lon)
                intersect_lat = s_lat + (e_lat - s_lat) * ratio

                folium.PolyLine(
                    [[s_lat, s_lon], [intersect_lat, edge_lon]],
                    color=self.route_color, weight=4, opacity=0.6
                ).add_to(self.m)
                folium.PolyLine(
                    [[intersect_lat, -edge_lon], [e_lat, e_lon]],
                    color=self.route_color, weight=4, opacity=0.6
                ).add_to(self.m)
            else:
                folium.PolyLine(
                    [[s_lat, s_lon], [e_lat, e_lon]],
                    color=self.route_color, weight=4, opacity=0.6
                ).add_to(self.m)

            # Popisok so vzdialenosťou/počtom letov
            if value_show:
                m_lat, m_lon = (s_lat + e_lat) / 2, (s_lon + v_e_lon) / 2
                final_m_lon = (m_lon + 180) % 360 - 180
                
                # Zobrazenie flight_count (ak existuje) alebo distance
                val = row.get("flight_count", row.get("distance", ""))
                label = f"{val}"
                
                folium.Marker(
                    [m_lat, final_m_lon],
                    icon=folium.DivIcon(html=f"""
                        <div style="color: {self.route_color}; font-weight: bold; font-size: 12pt; 
                                    white-space: nowrap; text-shadow: 1px 1px white; 
                                    background: rgba(255,255,255,0.6); padding: 1px 4px; border-radius: 3px;">
                            {label}
                        </div>"""),
                ).add_to(self.m)

    def showMap(self, filename="mapa_vysledok.html"):
        """Uloží mapu do HTML a otvorí ju v prehliadači."""
        self.m.save(filename)
        webbrowser.open(filename)

In [390]:
def make_start_end(edges_df):
    new_edges = edges_df.copy().reset_index()
    new_edges[["start", "end"]] = (
        new_edges[["# source", "target"]]
        .apply(
            lambda row: f"{nodes.loc[row['# source'],'latitude']},"
            + f"{nodes.loc[row['# source'],'longitude']};"
            + f"{nodes.loc[row['target'],'latitude']},"
            + f"{nodes.loc[row['target'],'longitude']}",
            axis=1,
        )
        .str.split(";", expand=True)
    )
    return new_edges

In [391]:
# 1. Zoznam indexov trasy
node_indices = [
    2317,
    2287,
    2330,
    2292,
    57,
    86,
    125,
    770,
    461,
    464,
    468,
    3205,
    466,
    465,
]

# 2. Manuálny mapping pre tieto konkrétne letiská (z tvojho predošlého zadania)
route_mapping = {
    2317: (79, 75),  # Peawanuck
    2287: (220, 42),  # Attawapiskat
    2330: (-20, 47),  # Kashechewan
    2292: (200, -2),  # Fort Albany
    57: (-20, 7),  # Moosonee
    86: (180, -15),  # Timmins
    125: (79, -10),  # Toronto (Pearson)
    770: (79, 75),  # Istanbul (Atatürk)
    461: (180, 40),  # Kinshasa (Ndjili)
    464: (200, 45),  # Kisangani
    468: (130, -5),  # Goma
    3205: (-10, 10),  # Beni
    466: (-5, 50),  # Bunia
    465: (90, 90),  # Isiro (Matari)
}

# 3. Vytvorenie DataFrame pre uzly (Nodes)
route_nodes = nodes.loc[node_indices].copy()
# 1. Priradenie kotvy cez index (idx je index v DataFrame, ktorý zodpovedá route_mapping)
route_nodes["anchor"] = route_nodes.index.map(route_mapping)

# 2. Ak by nejaký index v slovníku chýbal, dáme mu stredovú kotvu
route_nodes["anchor"] = route_nodes["anchor"].fillna(
    {i: (79, 40) for i in route_nodes.index}
)

# 3. Vykreslenie
rm = RouteMap()
rm.add_nodes(route_nodes, show_text=True)


route_edges = pd.DataFrame(
    zip(node_indices, node_indices[1:]), columns=["# source", "target"]
)
route_edges = edges.loc[zip(node_indices, node_indices[1:])]

route_edges = make_start_end(route_edges)
rm.add_route(route_edges, False)
rm.showMap("./vizs/route_map.html")

In [392]:
route_edges

,# source,target,distance,start,end
0,2317,2287,302.225602,"54.98809814453125,-85.44329833984375","52.9275016784668,-82.43190002441406"
1,2287,2330,87.987282,"52.9275016784668,-82.43190002441406","52.282501220703125,-81.67780303955078"
2,2330,2292,9.114073,"52.282501220703125,-81.67780303955078","52.20140075683594,-81.6968994140625"
3,2292,57,126.003148,"52.20140075683594,-81.6968994140625","51.29109954833984,-80.60780334472656"
4,57,86,307.661795,"51.29109954833984,-80.60780334472656","48.5696983337,-81.376701355"
5,86,125,560.551645,"48.5696983337,-81.376701355","43.6772003174,-79.63059997559999"
6,125,770,8201.411309,"43.6772003174,-79.63059997559999","40.976898,28.8146"
7,770,461,5225.914053,"40.976898,28.8146","-4.38575,15.4446"
8,461,464,1225.538438,"-4.38575,15.4446","0.481638997793,25.3379993439"
9,464,468,495.485122,"0.481638997793,25.3379993439","-1.670809984207153,29.238500595092773"


In [393]:
# 1. Zoznam indexov trasy
node_indices = [2909, 863, 860, 868, 1744, 1642, 1655, 3109]

# 2. Manuálny mapping pre tieto konkrétne letiská (z tvojho predošlého zadania)
route_mapping_2 = {
    2909: (79, 75),  # Sinop (Turkey) - Hore
    863: (-10, -0),  # Panama - Vľavo
    860: (20, 70),    # Bocas Del Toro - Vpravo
    868: (180, 0),   # San Jose (Costa Rica) - Hore
    1744: (79, 75),  # Houston (USA) - Hore
    1642: (79, 70),   # Beijing (China) - Vpravo
    1655: (185, 32), # Xi'an (China) - Vľavo
    3109: (79, 85),  # Mackenzie (Canada) - Hore
}

# 3. Vytvorenie DataFrame pre uzly (Nodes)
route_nodes_2 = nodes.loc[node_indices].copy()
# 1. Priradenie kotvy cez index (idx je index v DataFrame, ktorý zodpovedá route_mapping)
route_nodes_2["anchor"] = route_nodes_2.index.map(route_mapping_2)

# 2. Ak by nejaký index v slovníku chýbal, dáme mu stredovú kotvu
route_nodes_2["anchor"] = route_nodes_2["anchor"].fillna(
    {i: (79, 40) for i in route_nodes_2.index}
)

# 3. Vykreslenie
rm = RouteMap()
rm.add_nodes(route_nodes_2, show_text=True)


route_edges = edges.loc[zip(node_indices, node_indices[1:])]
route_edges = make_start_end(route_edges)
route_edges["src_city"] = route_edges["# source"].map(nodes["city"])
route_edges["dest_city"] = route_edges["target"].map(nodes["city"])

rm.add_route(route_edges, False)
rm.showMap("./vizs/distance_map.html")

In [394]:
route_edges

,# source,target,distance,start,end,src_city,dest_city
0,2909,863,11302.730990,"42.015800476074,35.066398620605","8.973340034484863,-79.55560302734375",Sinop,Panama
1,863,860,298.774291,"8.973340034484863,-79.55560302734375","9.340849876403809,-82.25080108642578",Panama,Bocas Del Toro
2,860,868,226.647594,"9.340849876403809,-82.25080108642578","9.993860244750977,-84.20880126953125",Bocas Del Toro,San Jose
3,868,1744,2505.695965,"9.993860244750977,-84.20880126953125","29.984399795532227,-95.34140014648438",San Jose,Houston
4,1744,1642,11559.932616,"29.984399795532227,-95.34140014648438","40.0801010131836,116.58499908447266",Houston,Beijing
5,1642,1655,933.833791,"40.0801010131836,116.58499908447266","34.447102,108.751999",Beijing,Xi'an
6,1655,3109,8887.536731,"34.447102,108.751999","55.304402,-123.132004",Xi'an,Mackenzie British Columbia


In [395]:
one_degree_nodes = pd.read_csv("data/one_degree.csv",index_col="index")
leaves = nodes.loc[pd.read_csv("data/leaf_airports.csv")["index"]]
leaves = leaves[leaves.index.isin(one_degree_nodes.index) == False]
one_degree_nodes = pd.concat([one_degree_nodes, leaves])
one_degree_nodes["way"] = one_degree_nodes["way"].fillna("leaf")
one_degree_nodes

,name,city,country,latitude,longitude,continent,degree,way
471,Kalemie Airport,Kalemie,Congo (Kinshasa),-5.875560,29.250000,Africa,1.0,in
488,Tan Tan Airport,Tan Tan,Morocco,28.448200,-11.161300,Africa,1.0,out
660,Filippos Airport,Kozani,Greece,40.286098,21.840799,Europe,1.0,in
721,Ovda International Airport,Ovda,Israel,29.940300,34.935799,Asia,1.0,out
1259,Teniente Coronel Luis a Mantilla Airport,Tulcan,Ecuador,0.809506,-77.708099,South America,1.0,in
...,...,...,...,...,...,...,...,...
3202,Zhangjiakou Ningyuan Airport,Zhangjiakou,China,40.738602,114.930000,Asia,NaN,leaf
3203,Arxan Yi'ershi Airport,Arxan,China,47.310600,119.911700,Asia,NaN,leaf
3210,Tarko-Sale Airport,Tarko-Sale,Russia,64.930801,77.818100,Europe,NaN,leaf
3212,Alashankou Bole (Bortala) airport,Bole,China,44.895000,82.300000,Asia,NaN,leaf


In [396]:
one_degree_nodes

,name,city,country,latitude,longitude,continent,degree,way
471,Kalemie Airport,Kalemie,Congo (Kinshasa),-5.875560,29.250000,Africa,1.0,in
488,Tan Tan Airport,Tan Tan,Morocco,28.448200,-11.161300,Africa,1.0,out
660,Filippos Airport,Kozani,Greece,40.286098,21.840799,Europe,1.0,in
721,Ovda International Airport,Ovda,Israel,29.940300,34.935799,Asia,1.0,out
1259,Teniente Coronel Luis a Mantilla Airport,Tulcan,Ecuador,0.809506,-77.708099,South America,1.0,in
...,...,...,...,...,...,...,...,...
3202,Zhangjiakou Ningyuan Airport,Zhangjiakou,China,40.738602,114.930000,Asia,NaN,leaf
3203,Arxan Yi'ershi Airport,Arxan,China,47.310600,119.911700,Asia,NaN,leaf
3210,Tarko-Sale Airport,Tarko-Sale,Russia,64.930801,77.818100,Europe,NaN,leaf
3212,Alashankou Bole (Bortala) airport,Bole,China,44.895000,82.300000,Asia,NaN,leaf


In [397]:
continent_one_degree = one_degree_nodes.groupby(["continent","way"])["name"].count().reset_index().rename(columns={"name":"count"})
continent_one_degree = continent_one_degree.pivot_table(index=["continent"],columns="way",values="count").fillna(0).reset_index()
continent_one_degree[["in","out","leaf"]] = continent_one_degree[["in","out","leaf"]].astype(int)
# Definujeme vizuálne stredy kontinentov (X, Y)
visual_centers = {
    "Europe": [71.0, 15.0],
    "Asia": [65.0, 115.0],
    "Africa": [-13, -5.0],
    "North America": [65.0, -90.0],
    "South America": [-30.0, -87.0],
    "Oceania": [-30.0, 98.0],
}


def get_visual_center(cont_name, shift=0):
    cont_name = str(cont_name).strip()
    if cont_name in visual_centers:
        coords = visual_centers[cont_name]
        return f"{coords[0] },{coords[1]+shift}"
    return None


# Aplikujeme tieto stredy na flow dataframe
continent_one_degree[["latitude", "longitude"]] = (
    continent_one_degree["continent"]
    .apply(get_visual_center, shift=-3)
    .str.split(",", expand=True)
)

intra_continent_df = continent_one_degree.dropna(subset=["longitude", "latitude"]).rename(
    columns={"continent": "name"})
continent_summary_place_mapping = {
    "Africa": "up",
    "Asia": "up",
    "Europe": "up",
    "North America": "up",
    "Oceania": "up",
    "South America": "up"
}
continent_summary_anchor_mapping = {
    "up": (79, 75),
    "bottom": (79, 0),
    "left": (160, 30),
    "right": (0, 32)
}
# 1. Priradíme smer (podľa stĺpca 'city', kde máš názov kontinentu)
intra_continent_df["place"] = intra_continent_df["name"].map(continent_summary_place_mapping).fillna("up")

# 2. Priradíme kotvu
intra_continent_df["anchor"] = intra_continent_df["place"].map(continent_summary_anchor_mapping)

In [398]:
one_degree_map = RouteMap()
one_degree_map.add_nodes(one_degree_nodes)
one_degree_map.add_nodes(intra_continent_df,True)
one_degree_map.add_categorical_legend()
one_degree_map.showMap()

In [399]:
components = [
    {931, 932, 933, 934, 935, 936, 937, 2535, 2537, 2536},
    {2426, 2971, 1837, 2970},
    {3023, 2643, 3022, 1903},
    {2072, 2300},
    {2259, 3116},
    {2905, 3097, 2380, 2382},
]
# Reprezentatívne uzly pre každý "ostrov"
island_summaries = {
    931:  "Nová Kaledónia",
    2426: "Regionálne siete USA",
    3023: "Vidiecke USA",
    2072: "Severná Kanada",
    2259: "Grónsko",
    2905: "Namíbia"
}

# Mapping kotiev pre tieto súhrnné body (aby boli pekne nad bodom)
summary_anchors = {i: ((79, 75) if i != 2426 else (79,-10)) for i in island_summaries.keys()}
# Zoznam všetkých indexov, ktoré patria do malých komponentov
all_island_indices = [idx for comp in components for idx in comp]

# Odfiltrovanie nodes pre ostrovy
island_nodes = nodes.loc[all_island_indices].copy()

# 1. Vyfiltrujeme len tie uzly, ktoré sme si vybrali ako "reprezentantov"
summary_nodes = nodes.loc[list(island_summaries.keys())].copy()

# 2. Prepíšeme ich názvy na názvy celých komponentov
summary_nodes["city"] = summary_nodes.index.map(island_summaries)
summary_nodes["country"] = summary_nodes.index.map({a:len(c) for c, a in zip(components, island_summaries.keys())})
summary_nodes["country"] = summary_nodes["country"].apply(lambda x: f"Počet letísk: {x}")

# 3. Pridáme kotvy
summary_nodes["anchor"] = summary_nodes.index.map(summary_anchors)
display(summary_nodes)
# Vykreslenie
rm_islands = RouteMap()
# Tu môžeš zmeniť farbu v add_nodes, ak chceš ostrovy odlíšiť
rm_islands.add_nodes(island_nodes)
rm_islands.add_nodes(summary_nodes, True)


# Ak chceš pridať aj cesty v rámci týchto ostrovov:
edges = edges.reset_index()
island_edges = edges[(edges["# source"].isin(all_island_indices) & edges["target"].isin(all_island_indices).values)].set_index(["# source", "target"])
island_edges = make_start_end(island_edges)
rm_islands.add_route(island_edges,False)

rm_islands.showMap("ostrovy_izolacie.html")

,name,city,country,latitude,longitude,continent,anchor
931,Koné Airport,Nová Kaledónia,Počet letísk: 10,-21.054300,164.837006,Oceania,"(79, 75)"
2426,William R Fairchild International Airport,Regionálne siete USA,Počet letísk: 4,48.120201,-123.500000,North America,"(79, -10)"
3023,Nikolski Air Station,Vidiecke USA,Počet letísk: 4,52.941601,-168.848999,North America,"(79, 75)"
2072,Victoria Harbour Seaplane Base,Severná Kanada,Počet letísk: 2,48.424986,-123.388867,North America,"(79, 75)"
2259,Neerlerit Inaat Airport,Grónsko,Počet letísk: 2,70.743103,-22.650499,North America,"(79, 75)"
2905,Katima Mulilo Airport,Namíbia,Počet letísk: 4,-17.634399,24.176701,Africa,"(79, 75)"


In [404]:
bridges = [(4, 2242),
 (4, 2249),
 (4, 2257),
 (7, 14),
 (9, 2265),
 (10, 14),
 (11, 14),
 (12, 14),
 (21, 113),
 (33, 88),
 (33, 131),
 (33, 2148),
 (34, 106),
 (34, 2308),
 (34, 3120),
 (37, 126),
 (38, 125),
 (54, 113),
 (57, 2292),
 (58, 3118),
 (63, 97),
 (69, 113),
 (78, 2298),
 (78, 3120),
 (93, 106),
 (95, 2311),
 (97, 116),
 (97, 127),
 (97, 2318),
 (97, 2332),
 (97, 3119),
 (103, 113),
 (121, 2290),
 (125, 128),
 (126, 2271),
 (126, 2310),
 (126, 2320),
 (126, 3045),
 (133, 138),
 (133, 150),
 (133, 2340),
 (142, 2338),
 (144, 2339),
 (156, 157),
 (156, 158),
 (156, 159),
 (156, 2341),
 (161, 165),
 (161, 166),
 (161, 172),
 (161, 173),
 (167, 170),
 (168, 170),
 (170, 3185),
 (183, 254),
 (184, 2906),
 (201, 205),
 (206, 597),
 (210, 618),
 (215, 216),
 (217, 218),
 (218, 220),
 (218, 223),
 (218, 225),
 (218, 227),
 (218, 232),
 (218, 233),
 (218, 372),
 (241, 261),
 (244, 2352),
 (254, 633),
 (270, 272),
 (270, 277),
 (270, 2344),
 (270, 2353),
 (275, 279),
 (279, 717),
 (295, 297),
 (295, 298),
 (295, 299),
 (307, 2203),
 (308, 327),
 (311, 315),
 (313, 315),
 (315, 323),
 (315, 324),
 (346, 369),
 (349, 368),
 (357, 364),
 (357, 368),
 (359, 368),
 (363, 368),
 (363, 2371),
 (381, 384),
 (381, 386),
 (381, 387),
 (381, 388),
 (381, 389),
 (381, 391),
 (381, 398),
 (381, 460),
 (397, 440),
 (402, 403),
 (402, 415),
 (409, 410),
 (409, 2373),
 (418, 421),
 (429, 433),
 (433, 439),
 (433, 2377),
 (446, 2378),
 (452, 453),
 (461, 2384),
 (462, 463),
 (467, 3209),
 (471, 3209),
 (482, 2385),
 (488, 492),
 (505, 2391),
 (508, 517),
 (522, 526),
 (529, 533),
 (533, 2398),
 (533, 2399),
 (533, 3108),
 (535, 538),
 (538, 539),
 (540, 541),
 (542, 544),
 (544, 2405),
 (544, 2896),
 (546, 548),
 (546, 3187),
 (549, 2407),
 (551, 2408),
 (551, 3072),
 (564, 2111),
 (570, 583),
 (570, 2888),
 (570, 2889),
 (578, 581),
 (578, 2470),
 (597, 641),
 (600, 611),
 (607, 629),
 (614, 625),
 (615, 629),
 (628, 721),
 (629, 637),
 (629, 638),
 (649, 1952),
 (651, 1952),
 (660, 1952),
 (666, 2126),
 (669, 1952),
 (692, 741),
 (692, 743),
 (698, 706),
 (718, 719),
 (718, 3117),
 (732, 734),
 (738, 739),
 (744, 750),
 (745, 770),
 (749, 750),
 (750, 752),
 (760, 2907),
 (760, 2910),
 (770, 2478),
 (770, 3168),
 (770, 3173),
 (797, 799),
 (797, 2086),
 (809, 832),
 (811, 832),
 (814, 832),
 (822, 832),
 (826, 1715),
 (827, 832),
 (831, 832),
 (832, 835),
 (832, 839),
 (832, 842),
 (832, 2485),
 (863, 3107),
 (863, 2486),
 (863, 2909),
 (866, 2490),
 (868, 2487),
 (868, 2934),
 (878, 879),
 (887, 1736),
 (888, 1736),
 (892, 898),
 (895, 898),
 (896, 898),
 (898, 2491),
 (902, 2494),
 (902, 2495),
 (902, 2496),
 (903, 2502),
 (904, 2051),
 (904, 2497),
 (904, 2499),
 (904, 2504),
 (907, 2506),
 (909, 1838),
 (918, 921),
 (930, 2514),
 (930, 2526),
 (930, 2528),
 (931, 934),
 (934, 935),
 (934, 936),
 (934, 2537),
 (939, 946),
 (939, 947),
 (939, 958),
 (939, 962),
 (939, 2505),
 (941, 944),
 (956, 959),
 (959, 961),
 (977, 988),
 (981, 982),
 (982, 987),
 (997, 1000),
 (1000, 2540),
 (1000, 2541),
 (1000, 2542),
 (1000, 2543),
 (1000, 2883),
 (1000, 2884),
 (1000, 3088),
 (1008, 2193),
 (1016, 1018),
 (1020, 1021),
 (1023, 1026),
 (1025, 1034),
 (1026, 1030),
 (1034, 2555),
 (1050, 1055),
 (1053, 1485),
 (1061, 1102),
 (1062, 1102),
 (1068, 1080),
 (1071, 1102),
 (1078, 1102),
 (1086, 1102),
 (1091, 1102),
 (1092, 1102),
 (1100, 1102),
 (1101, 1102),
 (1102, 1681),
 (1102, 2573),
 (1102, 2575),
 (1102, 2576),
 (1102, 3213),
 (1105, 2577),
 (1105, 2578),
 (1111, 1113),
 (1115, 1116),
 (1115, 1121),
 (1115, 1987),
 (1115, 2132),
 (1115, 2585),
 (1115, 2586),
 (1115, 2587),
 (1115, 2588),
 (1115, 2589),
 (1115, 2590),
 (1115, 2591),
 (1115, 2592),
 (1115, 2871),
 (1120, 2583),
 (1129, 1132),
 (1131, 1132),
 (1132, 1136),
 (1132, 1137),
 (1132, 1138),
 (1132, 1139),
 (1132, 1142),
 (1132, 1143),
 (1132, 1145),
 (1132, 1152),
 (1132, 1161),
 (1132, 3157),
 (1162, 3053),
 (1162, 3126),
 (1163, 1199),
 (1165, 1180),
 (1167, 1206),
 (1167, 3054),
 (1168, 2873),
 (1169, 3058),
 (1170, 1183),
 (1173, 1177),
 (1178, 3062),
 (1180, 1234),
 (1180, 2613),
 (1180, 3061),
 (1180, 3182),
 (1181, 1215),
 (1183, 1226),
 (1183, 1227),
 (1183, 3055),
 (1183, 3066),
 (1183, 3067),
 (1192, 2062),
 (1199, 3064),
 (1199, 3128),
 (1202, 3126),
 (1210, 1212),
 (1223, 1230),
 (1223, 2594),
 (1224, 2062),
 (1226, 1229),
 (1226, 3131),
 (1226, 3132),
 (1226, 3133),
 (1231, 2598),
 (1248, 3183),
 (1252, 1253),
 (1253, 2599),
 (1256, 1257),
 (1257, 1258),
 (1257, 1259),
 (1262, 2602),
 (1264, 1267),
 (1264, 1272),
 (1264, 1273),
 (1264, 1274),
 (1264, 1277),
 (1264, 1279),
 (1264, 1283),
 (1264, 1284),
 (1264, 1287),
 (1264, 1291),
 (1264, 1292),
 (1264, 1295),
 (1264, 1297),
 (1264, 1299),
 (1264, 2606),
 (1264, 3049),
 (1266, 1280),
 (1270, 1271),
 (1270, 1275),
 (1279, 2603),
 (1280, 2604),
 (1289, 1294),
 (1295, 2877),
 (1303, 1304),
 (1303, 3046),
 (1307, 2607),
 (1307, 2608),
 (1313, 1317),
 (1314, 1317),
 (1315, 1317),
 (1316, 1317),
 (1317, 1319),
 (1317, 1322),
 (1317, 1326),
 (1317, 2609),
 (1317, 2610),
 (1317, 3181),
 (1330, 1340),
 (1332, 1344),
 (1333, 1340),
 (1334, 1340),
 (1336, 1340),
 (1338, 1340),
 (1340, 1342),
 (1340, 1345),
 (1340, 1348),
 (1340, 1350),
 (1344, 1346),
 (1352, 2177),
 (1361, 1363),
 (1363, 2920),
 (1370, 1371),
 (1388, 2624),
 (1388, 2625),
 (1388, 2626),
 (1393, 1437),
 (1395, 3150),
 (1399, 2635),
 (1399, 2928),
 (1399, 3160),
 (1401, 2630),
 (1405, 2631),
 (1408, 2931),
 (1429, 2221),
 (1435, 2930),
 (1437, 2661),
 (1437, 2846),
 (1445, 1446),
 (1445, 1450),
 (1445, 1454),
 (1445, 1459),
 (1445, 1494),
 (1447, 1513),
 (1463, 2669),
 (1463, 3171),
 (1464, 3171),
 (1466, 2063),
 (1471, 2670),
 (1478, 1484),
 (1480, 1484),
 (1484, 2673),
 (1490, 1493),
 (1493, 1498),
 (1493, 1499),
 (1493, 2878),
 (1493, 2891),
 (1506, 2053),
 (1506, 2097),
 (1506, 2098),
 (1508, 1509),
 (1509, 1510),
 (1509, 1511),
 (1509, 2117),
 (1509, 2118),
 (1509, 2674),
 (1509, 2675),
 (1509, 2676),
 (1512, 1516),
 (1513, 1514),
 (1513, 3100),
 (1526, 1529),
 (1526, 1531),
 (1526, 1535),
 (1526, 1537),
 (1526, 1539),
 (1526, 1540),
 (1526, 1946),
 (1526, 1947),
 (1526, 2092),
 (1526, 2093),
 (1526, 2172),
 (1526, 2680),
 (1528, 1919),
 (1542, 2686),
 (1545, 2106),
 (1545, 2107),
 (1549, 1557),
 (1551, 2690),
 (1551, 2691),
 (1552, 1553),
 (1557, 2690),
 (1558, 1568),
 (1558, 1570),
 (1558, 3194),
 (1562, 1563),
 (1562, 1564),
 (1562, 3197),
 (1567, 1920),
 (1567, 3195),
 (1567, 3196),
 (1574, 1576),
 (1579, 1581),
 (1583, 1584),
 (1589, 1929),
 (1599, 1680),
 (1600, 1607),
 (1608, 1612),
 (1610, 1985),
 (1610, 2141),
 (1612, 1620),
 (1612, 2728),
 (1614, 2729),
 (1621, 1639),
 (1623, 2723),
 (1623, 2758),
 (1625, 2723),
 (1626, 2720),
 (1627, 2034),
 (1627, 2714),
 (1627, 2716),
 (1627, 2749),
 (1627, 2750),
 (1627, 2755),
 (1627, 2764),
 (1631, 1633),
 (1633, 1634),
 (1633, 2724),
 (1633, 2748),
 (1633, 2751),
 (1633, 3199),
 (1634, 2752),
 (1639, 1640),
 (1639, 2042),
 (1639, 2734),
 (1639, 2741),
 (1639, 2915),
 (1639, 2916),
 (1652, 2784),
 (1656, 2794),
 (1656, 2795),
 (1656, 2796),
 (1656, 2797),
 (1656, 3077),
 (1656, 3089),
 (1658, 2800),
 (1658, 3081),
 (1658, 3085),
 (1667, 3080),
 (1671, 3158),
 (1671, 3174),
 (1674, 1675),
 (1675, 2827),
 (1675, 2829),
 (1675, 2885),
 (1675, 2935),
 (1675, 3177),
 (1675, 3212),
 (1679, 2579),
 (1684, 1815),
 (1685, 1752),
 (1688, 2987),
 (1689, 2565),
 (1692, 1861),
 (1693, 1765),
 (1695, 2952),
 (1696, 1702),
 (1697, 1715),
 (1698, 2948),
 (1699, 1734),
 (1699, 1821),
 (1699, 2012),
 (1699, 2013),
 (1699, 2457),
 (1699, 2460),
 (1699, 2844),
 (1702, 1742),
 (1704, 2461),
 (1708, 2155),
 (1710, 1800),
 (1712, 1825),
 (1712, 1877),
 (1712, 2066),
 (1715, 2058),
 (1715, 2969),
 (1715, 2991),
 (1715, 2992),
 (1716, 2079),
 (1718, 2452),
 (1719, 2994),
 (1719, 2995),
 (1723, 1800),
 (1730, 1779),
 (1733, 1853),
 (1735, 1861),
 (1735, 3003),
 (1735, 3005),
 (1738, 1881),
 (1738, 2009),
 (1738, 2156),
 (1738, 2431),
 (1738, 2451),
 (1738, 2966),
 (1738, 2967),
 (1743, 1907),
 (1744, 1852),
 (1744, 2465),
 (1753, 1999),
 (1754, 2414),
 (1754, 2468),
 (1754, 2927),
 (1758, 1854),
 (1761, 1806),
 (1763, 2006),
 (1765, 2978),
 (1765, 2979),
 (1767, 2008),
 (1772, 1800),
 (1782, 1861),
 (1789, 2424),
 (1789, 2453),
 (1789, 2918),
 (1791, 1901),
 (1798, 1861),
 (1799, 1806),
 (1800, 1807),
 (1800, 1820),
 (1800, 1824),
 (1800, 1832),
 (1800, 1844),
 (1800, 1859),
 (1800, 2154),
 (1800, 2207),
 (1806, 2423),
 (1806, 2438),
 (1806, 2464),
 (1806, 2921),
 (1809, 1846),
 (1809, 2153),
 (1809, 2159),
 (1809, 2194),
 (1809, 2411),
 (1809, 2420),
 (1809, 2447),
 (1809, 2450),
 (1809, 2463),
 (1809, 2466),
 (1813, 2412),
 (1815, 3011),
 (1829, 2444),
 (1834, 2926),
 (1836, 1907),
 (1837, 2426),
 (1850, 2954),
 (1853, 1871),
 (1853, 2151),
 (1853, 2196),
 (1853, 2197),
 (1853, 2418),
 (1853, 2439),
 (1853, 2441),
 (1853, 2459),
 (1853, 2924),
 (1853, 2925),
 (1853, 2953),
 (1853, 2962),
 (1854, 1993),
 (1855, 1885),
 (1861, 1897),
 (1861, 1908),
 (1861, 2559),
 (1861, 2879),
 (1866, 2115),
 (1868, 3001),
 (1868, 2892),
 (1868, 2894),
 (1868, 3000),
 (1871, 2467),
 (1872, 1914),
 (1875, 2997),
 (1875, 2998),
 (1875, 3028),
 (1885, 1890),
 (1885, 2082),
 (1885, 2229),
 (1885, 2413),
 (1885, 2427),
 (1885, 2430),
 (1885, 2446),
 (1886, 2988),
 (1886, 3038),
 (1886, 3039),
 (1897, 2893),
 (1897, 3026),
 (1897, 3027),
 (1901, 2167),
 (1901, 2410),
 (1903, 2643),
 (1903, 3022),
 (1903, 3023),
 (1913, 1916),
 (1913, 2445),
 (1913, 2941),
 (1919, 1950),
 (1919, 2094),
 (1926, 2703),
 (1937, 2693),
 (1940, 1951),
 (1944, 2056),
 (1944, 2110),
 (1952, 2123),
 (1952, 2124),
 (1952, 2125),
 (1952, 2471),
 (1967, 2016),
 (1968, 1969),
 (1973, 1980),
 (1977, 1979),
 (1978, 1979),
 (1988, 2267),
 (1988, 2268),
 (1990, 3076),
 (1991, 2738),
 (1991, 2739),
 (2006, 3093),
 (2008, 2434),
 (2008, 2959),
 (2008, 2963),
 (2008, 2964),
 (2016, 2634),
 (2016, 2638),
 (2016, 2639),
 (2016, 2640),
 (2016, 2665),
 (2016, 2666),
 (2016, 2872),
 (2016, 3146),
 (2016, 3148),
 (2048, 2230),
 (2048, 2233),
 (2048, 2236),
 (2048, 2237),
 (2072, 2300),
 (2085, 2142),
 (2115, 2454),
 (2135, 2582),
 (2135, 2947),
 (2147, 2530),
 (2183, 2479),
 (2193, 2913),
 (2201, 2351),
 (2213, 2632),
 (2222, 2637),
 (2259, 3116),
 (2265, 2267),
 (2268, 3144),
 (2278, 2293),
 (2287, 2317),
 (2287, 2330),
 (2292, 2330),
 (2380, 2382),
 (2391, 2392),
 (2393, 3069),
 (2463, 2641),
 (2480, 2481),
 (2517, 2525),
 (2521, 2525),
 (2525, 2898),
 (2562, 2567),
 (2562, 2853),
 (2565, 3034),
 (2566, 2858),
 (2567, 2974),
 (2567, 2975),
 (2567, 3037),
 (2633, 3159),
 (2644, 3210),
 (2698, 2699),
 (2717, 2761),
 (2717, 2762),
 (2721, 2738),
 (2729, 3198),
 (2737, 2738),
 (2761, 2993),
 (2767, 3075),
 (2771, 3203),
 (2773, 3202),
 (2790, 3104),
 (2793, 2794),
 (2851, 3178),
 (2867, 2987),
 (2895, 3029),
 (2914, 2916),
 (2972, 3024),
 (2973, 3019),
 (3004, 3005),
 (3052, 3058),
 (3052, 3129),
 (3057, 3129),
 (3125, 3127)]

In [407]:
# 2. Extrakcia unikátnych letísk, ktoré tvoria mosty
unique_bridge_nodes = list(set([n for edge in bridges for n in edge]))

# 3. Príprava dát pre add_nodes
# Predpokladáme, že máš DataFrame 'df_airports' indexovaný pomocou ID letiska
df_bridge_nodes = nodes.loc[unique_bridge_nodes].copy()

# Ak chceš, aby fungovala legenda (way), priradíme im kategóriu
# Pre mosty sú to v podstate všetko tranzitné uzly v rámci globálnej kostry
df_bridge_nodes['way'] = 'transit' 

# 4. Príprava dát pre add_route
bridge_routes_list = []
for u, v in bridges:
    try:
        # Získanie koordinátov pre štart a cieľ
        start_coords = f"{nodes.loc[u, 'latitude']},{nodes.loc[u, 'longitude']}"
        end_coords = f"{nodes.loc[v, 'latitude']},{nodes.loc[v, 'longitude']}"
        
        bridge_routes_list.append({
            'start': start_coords,
            'end': end_coords,
            'flight_count': 1 # Most je v topológii jedna unikátna hrana
        })
    except KeyError:
        # Ošetrenie, ak by nejaké ID letiska chýbalo v tvojom df_airports
        continue

df_bridge_routes = pd.DataFrame(bridge_routes_list)

# 5. Samotná vizualizácia
mapa_mostov = RouteMap(center=[20, 0], zoom_start=2)

# Zmeníme farbu na oranžovú/červenú, aby kritické mosty poriadne "svietili"
mapa_mostov.route_color = "#8C361C" 

# Pridáme uzly (bez textu, aby mapa nebola preplnená tisíckami názvov)
mapa_mostov.add_nodes(df_bridge_nodes, show_text=False)

# Pridáme cesty (mosty)
mapa_mostov.add_route(df_bridge_routes, value_show=False)

# Uložíme a zobrazíme
mapa_mostov.showMap("mapa_kritickych_mostov.html")